# Lab 8: An agent, step by step, with a human approval gate
**Module 4.2: Agent patterns.**  **Time:** 30 minutes.

**Goal:** an *agent* is just a loop: the model picks a tool, your code runs it, the result goes back, repeat
until the model answers. You will read every step of the trace and add a **human approval** step for a
risky action (refunds above the agent's limit).

In [ ]:
# SETUP: run this cell first. It works in Google Colab and in VS Code.
import os, sys, subprocess
REPO_URL = "https://github.com/cybertide-solutions/ai-platform-architect-labs.git"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/course"):
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/course"], check=True)
    os.chdir("/content/course")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                      # VS Code starts in notebooks/; the course files are one level up
sys.path.insert(0, os.getcwd())
import labkit as lk
lk.setup();
import json
qc, _ = lk.build_policy_index()

## 1. Three tools

In [ ]:
ORDERS = {"ORD-1001": {"status": "shipped", "amount_inr": 2499},
          "ORD-1002": {"status": "delivered", "amount_inr": 7999},
          "ORD-1003": {"status": "processing", "amount_inr": 18999}}
REFUNDS = []

def order_status(order_id: str) -> dict:
    return ORDERS.get(order_id.upper(), {"error": "not found"})

def search_policies(query: str) -> list:
    return [{"doc": h["doc"], "page": h["page"], "text": h["text"]} for h in lk.search(qc, query, k=2)]

def issue_refund(order_id: str, amount_inr: int, reason: str) -> dict:
    REFUNDS.append({"order_id": order_id, "amount_inr": amount_inr, "reason": reason})
    return {"refund_id": f"RF-{len(REFUNDS):04d}", "status": "refund issued"}

TOOLS = {
    "order_status": (order_status, lk.tool_spec(order_status, "Get status and amount of an order.",
                     {"order_id": {"type": "string"}})),
    "search_policies": (search_policies, lk.tool_spec(search_policies, "Search company policies, e.g. refund rules.",
                        {"query": {"type": "string"}})),
    "issue_refund": (issue_refund, lk.tool_spec(issue_refund, "Issue a refund to the customer. This moves money.",
                     {"order_id": {"type": "string"}, "amount_inr": {"type": "integer"}, "reason": {"type": "string"}})),
}

## 2. The agent loop (this is the whole idea)
Read this cell slowly. Frameworks such as CrewAI, LangGraph or Google ADK wrap this same loop.

In [ ]:
def agent(user_msg, approval_needed=lambda name, args: False, max_steps=6):
    msgs = [{"role": "system", "content": "You are a customer support agent for Meridian Retail. "
             "Check the order and the refund policy before acting. Be brief."},
            {"role": "user", "content": user_msg}]
    specs = [spec for _, spec in TOOLS.values()]
    for step in range(1, max_steps + 1):
        msg = lk.chat(messages=msgs, tools=specs, tag="agent")
        if not msg.tool_calls:                                   # no tool wanted: we are done
            print(f"[step {step}] FINAL: {msg.content}")
            return msg.content
        msgs.append({"role": "assistant", "content": msg.content or "", "tool_calls": [
            {"id": c.id, "type": "function", "function": {"name": c.function.name, "arguments": c.function.arguments}}
            for c in msg.tool_calls]})
        for c in msg.tool_calls:
            name, args = c.function.name, json.loads(c.function.arguments or "{}")
            print(f"[step {step}] MODEL WANTS: {name}({args})")
            if approval_needed(name, args) and not lk.console_approver(name, args):
                result = {"error": "Rejected by a human reviewer. Tell the customer a team lead will follow up."}
            else:
                result = TOOLS[name][0](**args)
            print(f"[step {step}] RESULT: {str(result)[:150]}")
            msgs.append({"role": "tool", "tool_call_id": c.id, "content": json.dumps(result)})
    return "step limit reached"

## 3. Run it without any control

In [ ]:
agent("My order ORD-1002 arrived damaged. Please refund the full amount.")
print("Refunds issued so far:", REFUNDS)

The policy says agents may approve refunds up to INR 5,000 on their own. Did the agent respect that?
Even if it did this time, **would you rely on the model to always respect it?**

## 4. Add the approval gate in code, not in the prompt
You will be asked to type `yes` or `no` when the gate triggers.

In [ ]:
def needs_approval(name, args):
    return name == "issue_refund" and int(args.get("amount_inr", 0)) > 5000

REFUNDS.clear()
agent("My order ORD-1002 arrived damaged. Please refund the full amount.", approval_needed=needs_approval)
print("Refunds issued:", REFUNDS)

## 5. The same job as a fixed workflow
No model decides the steps. The model is used only where language is needed.

In [ ]:
def refund_workflow(order_id: str, reason: str) -> str:
    order = order_status(order_id)
    if "error" in order:
        return "Order not found."
    if order["amount_inr"] <= 5000:
        r = issue_refund(order_id, order["amount_inr"], reason)
        return lk.chat(f"Tell the customer politely in 2 sentences that refund {r['refund_id']} "
                       f"of INR {order['amount_inr']} was issued.", tag="workflow")
    return "Escalated to a team lead (amount above INR 5,000)."

print(refund_workflow("ORD-1001", "damaged"))
print(refund_workflow("ORD-1002", "damaged"))

## Discuss
* Workflow or agent? The workflow is predictable, testable and cheap. The agent handles messy requests.
  Most production systems are workflows with small agentic parts.
* Where else does your organisation need a human approval step? Who is the approver, and what is the audit record?
* Optional extras: `lab08b_crewai.ipynb` and `lab08c_google_adk.ipynb` build this same agent with two frameworks.